# 03 – LLM extraction (MedGemma 1.5 and Gemma 3)

Asks a language model, zero-shot, for the Gleason pattern pair in each report. The same
notebook runs both models: only `model_id` in the config changes, so the prompt, the decoding
settings and the parsing are identical (`common.py` does the parsing, scoring and logging).

**Needs a GPU.** Run it in a GPU session on IDUN, not on the login node.

Notes on the setup:
- Greedy decoding (`do_sample=False`), so repeated runs give the same output.
- The whole prompt (instructions + report) is one text part in one user message, because the
  chat template joins several text parts with nothing in between.
- MedGemma 1.5 writes reasoning between `<unused94>thought` and `<unused95>` before its answer.
  `block_thinking=True` bans the `<unused94>` token so it answers directly. Both models share
  the same vocabulary, so the ban is applied the same way to both.
- The raw output is decoded with the special tokens kept, so the reasoning markers stay visible
  in the saved output. The parser reads the text after `<unused95>`.
- No truncation: the longest prompt (instructions + report) is 2,834 tokens with the Gemma 3
  tokenizer, far below the context length, so every report is sent in full to both models.

In [9]:
import json
import sys
import time

import pandas as pd
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForImageTextToText, AutoProcessor

sys.path.insert(0, ".")  # common.py lives next to this notebook
import common

In [ ]:
config = {
    "method": "llm",
    "model_id": "google/medgemma-1.5-4b-it",  # "google/medgemma-1.5-4b-it" or "google/gemma-3-4b-it"
    "subset": "tuning",  # "tuning" or "eval"
    "n_reports": None,  # None = all reports in the subset; a number for a quick test
    # The prompt. "{report}" is replaced with the report text as it is (no cleaning).
    # Unlike the regex (hard-coded "highest" rule), the prompt describes which score is wanted:
    # the overall score of the examined specimen, as in the final diagnosis.
    "prompt_template": (
        "Below is a pathology report for a patient with prostate cancer. The text was produced by "
        "OCR from a scanned document and may contain typos and noise.\n\n"
        "Find the Gleason score of the prostate cancer in the report. A Gleason score consists of a "
        "primary pattern and a secondary pattern, each graded from 1 to 5. It is often written as "
        "\"primary + secondary = total\", for example \"3 + 4 = 7\".\n\n"
        "The report may mention more than one Gleason score, for example from an earlier biopsy, in "
        "the clinical history, or for separate tumor nodules. Report the overall Gleason score of the "
        "tumor in the specimen examined in this report, as given in the final diagnosis. Do not use "
        "scores from earlier biopsies or the clinical history. If separate nodules are graded, use the "
        "score of the dominant tumor. Ignore tertiary patterns.\n\n"
        "If the report does not contain a Gleason score, use null for both patterns.\n\n"
        "Answer with JSON only, in exactly this format:\n"
        "{\"primary\": <primary pattern>, \"secondary\": <secondary pattern>}\n\n"
        "Report:\n"
        "{report}"
    ),
    "max_new_tokens": 1024,  # same for both models; high enough for MedGemma's reasoning
    "do_sample": False,  # greedy decoding
    "block_thinking": True,  # True: ban <unused94>, so MedGemma cannot reason. Main comparison: True for both models
    "dtype": "bfloat16",
    "seed": 13,  # greedy decoding does not sample, but fix the seed anyway
    "log_run": True,
    "note": "",
}

## Download the weights

Downloads the model to the default Hugging Face cache (`~/.cache/huggingface`) if it is not
there yet, about 8.6 GB per model. If it is already there (MedGemma, or Gemma 3 after the first
run) this only checks the files and takes a few seconds. Both models are gated: the Hugging Face
token on IDUN must belong to an account that has accepted the model's license.

In [11]:
model_path = snapshot_download(config["model_id"])
print("Weights in", model_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Weights in /cluster/home/hermaos/.cache/huggingface/hub/models--google--gemma-3-4b-it/snapshots/093f9f388b31de276ce2de164bdc2081324b9767


## Load the model

Loaded from the cache filled above. Both models are Gemma 3 image-text models, so they are
loaded with the same classes; only text is sent.

In [12]:
torch.manual_seed(config["seed"])
processor = AutoProcessor.from_pretrained(config["model_id"])
model = AutoModelForImageTextToText.from_pretrained(
    config["model_id"],
    dtype=getattr(torch, config["dtype"]),
    device_map="auto",
)
model.eval()

# Token that starts MedGemma's reasoning block, banned when block_thinking is True
thought_token_id = processor.tokenizer.convert_tokens_to_ids("<unused94>")
print("Model on", model.device, "| <unused94> id:", thought_token_id)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model on cuda:0 | <unused94> id: 100


## Prompt and generation

In [13]:
def build_messages(report_text, config):
    """Build the chat for one report.

    Args:
        report_text: the OCR text of the report, used as it is.
        config: the config dict (prompt_template).

    Returns:
        A list with one user message whose content is a single text part.
    """
    # str.replace, not str.format: the template contains JSON braces
    prompt = config["prompt_template"].replace("{report}", report_text)
    return [{"role": "user", "content": [{"type": "text", "text": prompt}]}]


def generate(report_text, config):
    """Run the model on one report.

    Args:
        report_text: the OCR text of the report.
        config: the config dict (prompt and decoding settings).

    Returns:
        dict with raw_output (the generated text, special tokens kept), n_prompt_tokens,
        n_new_tokens, hit_token_limit (True if generation stopped at max_new_tokens, so the
        answer may be cut off) and seconds.
    """
    inputs = processor.apply_chat_template(
        build_messages(report_text, config),
        add_generation_prompt=True,  # end with "<start_of_turn>model", the model's turn
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    n_prompt = inputs["input_ids"].shape[-1]

    start = time.time()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=config["max_new_tokens"],
            do_sample=config["do_sample"],
            # bad_words_ids: token sequences the model may never write (None = no ban)
            bad_words_ids=[[thought_token_id]] if config["block_thinking"] else None,
        )
    seconds = time.time() - start

    new_tokens = output[0][n_prompt:]
    # Keep special tokens so <unused94>/<unused95> stay in the raw output; drop the end markers
    raw = processor.decode(new_tokens, skip_special_tokens=False)
    for marker in ["<end_of_turn>", "<eos>"]:
        raw = raw.replace(marker, "")
    return {
        "raw_output": raw.strip(),
        "n_prompt_tokens": n_prompt,
        "n_new_tokens": len(new_tokens),
        "hit_token_limit": len(new_tokens) >= config["max_new_tokens"],
        "seconds": round(seconds, 2),
    }

## Quick look at one report

Shows the full prompt and the raw answer for the first report of the subset, to check the
setup before starting a full run.

In [14]:
data = common.load_data(config["subset"], n=config["n_reports"])
example = data.iloc[0]
print(build_messages(example["text"], config)[0]["content"][0]["text"][:1500], "...\n")
result = generate(example["text"], config)
print(result)
print("Parsed:", common.parse_answer(result["raw_output"]))
print("Label:", f"{example['label_primary']}+{example['label_secondary']}")

Below is a pathology report for a patient with prostate cancer. The text was produced by OCR from a scanned document and may contain typos and noise.

Find the Gleason score of the prostate cancer in the report. A Gleason score consists of a primary pattern and a secondary pattern, each graded from 1 to 5. It is often written as "primary + secondary = total", for example "3 + 4 = 7".

The report may mention more than one Gleason score, for example from an earlier biopsy, in the clinical history, or for separate tumor nodules. Report the overall Gleason score of the tumor in the specimen examined in this report, as given in the final diagnosis. Do not use scores from earlier biopsies or the clinical history. If separate nodules are graded, use the score of the dominant tumor. Ignore tertiary patterns.

If the report does not contain a Gleason score, use null for both patterns.

Answer with JSON only, in exactly this format:
{"primary": <primary pattern>, "secondary": <secondary pattern>

## Run on the whole subset

In [15]:
results = []
for i, report in enumerate(data.itertuples(), start=1):
    result = generate(report.text, config)
    results.append(result)
    parsed = common.parse_answer(result["raw_output"])
    print(f"{i}/{len(data)} {report.patient_id}: {parsed['status']} "
          f"{parsed['pred_primary']}+{parsed['pred_secondary']} "
          f"(label {report.label_primary}+{report.label_secondary}), "
          f"{result['n_new_tokens']} tokens, {result['seconds']} s")

extra = {key: [r[key] for r in results] for key in ["n_prompt_tokens", "n_new_tokens", "hit_token_limit", "seconds"]}
table = common.build_run_table(data, [r["raw_output"] for r in results], config, extra=extra)
scores = common.score_run(table)
# Generation statistics, saved with the scores
scores["hit_token_limit"] = int(table["hit_token_limit"].sum())
scores["mean_new_tokens"] = round(float(table["n_new_tokens"].mean()), 1)
scores["max_prompt_tokens"] = int(table["n_prompt_tokens"].max())
scores["total_minutes"] = round(float(table["seconds"].sum()) / 60, 1)

print(json.dumps({k: v for k, v in scores.items() if k != "confusion_matrix"}, indent=2))
print("\nConfusion matrix (rows: true score, columns: predicted):")
print(pd.DataFrame(scores["confusion_matrix"]).T.fillna(0).astype(int).sort_index(axis=1))

1/30 TCGA-CH-5763: valid 3+4 (label 3+4), 13 tokens, 0.58 s
2/30 TCGA-CH-5792: valid 4+5 (label 4+5), 13 tokens, 0.59 s
3/30 TCGA-EJ-5494: valid 4+3 (label 4+3), 18 tokens, 0.81 s
4/30 TCGA-EJ-5522: valid 3+4 (label 3+4), 18 tokens, 0.83 s
5/30 TCGA-EJ-7797: valid 3+4 (label 3+4), 18 tokens, 0.81 s
6/30 TCGA-EJ-A46G: valid 4+4 (label 4+4), 13 tokens, 0.6 s
7/30 TCGA-FC-7708: valid 4+3 (label 4+3), 18 tokens, 0.81 s
8/30 TCGA-FC-A6HD: valid 4+3 (label 4+3), 18 tokens, 0.81 s
9/30 TCGA-G9-6369: valid 4+3 (label 4+3), 18 tokens, 0.88 s
10/30 TCGA-HC-7075: valid 3+3 (label 3+3), 18 tokens, 0.8 s
11/30 TCGA-HC-7232: valid 4+5 (label 4+5), 18 tokens, 0.87 s
12/30 TCGA-HC-7738: valid 3+4 (label 3+4), 18 tokens, 0.86 s
13/30 TCGA-KC-A7FA: valid 3+4 (label 3+4), 18 tokens, 0.93 s
14/30 TCGA-KK-A59V: invalid 9+5 (label 4+5), 18 tokens, 0.93 s
15/30 TCGA-KK-A59Z: valid 4+3 (label 4+3), 18 tokens, 0.86 s
16/30 TCGA-KK-A6E8: valid 4+5 (label 4+5), 18 tokens, 0.89 s
17/30 TCGA-KK-A7AZ: valid 4+3 (la

## Errors

In [16]:
errors = table[~table["correct_pair"]].copy()
errors["label"] = errors["label_primary"].astype(str) + "+" + errors["label_secondary"].astype(str)
errors["predicted"] = errors["pred_primary"].astype(str) + "+" + errors["pred_secondary"].astype(str)
print(f"{len(errors)} of {len(table)} pattern pairs wrong")
errors[["patient_id", "status", "parse_error", "label", "predicted", "possible_hallucination", "raw_output"]]

3 of 30 pattern pairs wrong


,patient_id,status,parse_error,label,predicted,possible_hallucination,raw_output
13,TCGA-KK-A59V,invalid,pattern_out_of_range,4+5,9+5,False,"```json\n{""primary"": 9, ""secondary"": 5}\n```"
18,TCGA-KK-A8IH,invalid,pattern_out_of_range,4+3,7+3,False,"```json\n{""primary"": 7, ""secondary"": 3}\n```"
24,TCGA-X4-A8KQ,valid,NaN,4+5,4+3,False,"```json\n{""primary"": 4, ""secondary"": 3}\n```"


## Save

In [17]:
common.log_run(config, table, scores)

Saved results/tuning/v1
Changes since the previous run:
  - first run


'v1'